In [ ]:
from __future__ import annotations

from pprint import pprint

import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

from blackhole_paradox.protocols.yoshida_yao import run_yoshida_yao_protocol
from blackhole_paradox.visualization.trace_render import (
    plot_chunk_metrics,
    stage_lookup,
    trace_timeline_rows,
)


def render_trace(trace: dict) -> None:
    display(Markdown("## Simulation Trace"))
    display(Markdown("### Timeline"))
    for row in trace_timeline_rows(trace):
        display(Markdown(f"- **{row.title}** (`{row.stage}`): {row.description}"))

    display(Markdown("### Recovery Summary"))
    pprint(
        {
            "recovered_message": trace.get("recovered_message"),
            "mean_fidelity": trace.get("metrics", {}).get("mean_fidelity"),
            "mean_bitwise_accuracy": trace.get("metrics", {}).get("mean_bitwise_accuracy"),
            "mean_success_probability": trace.get("metrics", {}).get("mean_success_probability"),
        }
    )

    display(Markdown("### Per-chunk Bitstrings"))
    for idx, (enc, rec) in enumerate(
        zip(trace.get("encoded_blocks", []), trace.get("recovered_blocks", []))
    ):
        display(Markdown(f"- chunk {idx}: encoded `{enc}` -> recovered `{rec}`"))

    fig = plot_chunk_metrics(trace)
    display(fig)

    for stage_name in [
        "encoding",
        "scrambling",
        "entanglement_growth",
        "radiation_split",
        "noise",
        "decoding",
    ]:
        stage = stage_lookup(trace, stage_name)
        if stage:
            display(Markdown(f"### Stage Data: {stage.get('title', stage_name)}"))
            pprint(stage.get("data", {}))

In [ ]:
message = widgets.Text(
    value="hello quantum world",
    description="Message:",
    layout=widgets.Layout(width="600px"),
)

mode = widgets.ToggleButtons(
    options=["explain", "research"],
    value="explain",
    description="Mode:",
)

message_qubits = widgets.IntSlider(value=8, min=4, max=16, step=1, description="M qubits")
black_hole_qubits = widgets.IntSlider(value=6, min=2, max=12, step=1, description="BH qubits")
reference_qubits = widgets.IntSlider(value=6, min=2, max=12, step=1, description="Ref qubits")
radiation_qubits = widgets.IntSlider(value=4, min=1, max=12, step=1, description="Rad qubits")
scramble_depth = widgets.IntSlider(value=3, min=1, max=8, step=1, description="Depth")
max_chunks = widgets.IntSlider(value=8, min=1, max=32, step=1, description="Max chunks")
noise_model = widgets.Dropdown(
    options=["none", "depolarizing", "thermal"],
    value="none",
    description="Noise",
)
noise_strength = widgets.FloatSlider(
    value=0.01,
    min=0.0,
    max=0.2,
    step=0.005,
    readout_format=".3f",
    description="Strength",
)

run_button = widgets.Button(description="Run Simulation", button_style="primary")
out = widgets.Output()


def run_clicked(_):
    with out:
        clear_output(wait=True)
        trace = run_yoshida_yao_protocol(
            message=message.value,
            message_qubits=message_qubits.value,
            black_hole_qubits=black_hole_qubits.value,
            reference_qubits=reference_qubits.value,
            radiation_qubits=radiation_qubits.value,
            scramble_depth=scramble_depth.value,
            mode=mode.value,
            max_chunks=max_chunks.value,
            noise_model=noise_model.value,
            noise_strength=noise_strength.value,
        )
        render_trace(trace)


run_button.on_click(run_clicked)

controls = widgets.VBox(
    [
        message,
        widgets.HBox([mode, noise_model, noise_strength]),
        widgets.HBox([message_qubits, black_hole_qubits, reference_qubits]),
        widgets.HBox([radiation_qubits, scramble_depth, max_chunks]),
        run_button,
    ]
)

display(controls, out)